# Breadboard recognition: first training experiment
Upload **breadboard-pilot-ready.zip** when prompted, not the original Roboflow ZIP. Choose a T4 GPU runtime, then run each code cell from top to bottom.

This notebook matches BME Electronics v5: **breadboard, resistor, wire**, with **8 keypoint slots**. There are **no LEDs**. This is a small pilot, not a finished circuit-debugging model.

The 27 photos were grouped by visually related layouts: 21 training, 3 validation, 3 test. All related red/black wire photos stay together in training; consequently validation/test have **no wire examples**. Their metrics evaluate breadboards/resistors only and cannot establish wire accuracy.

574 boxes were expanded to include their labeled points or give very thin boxes a margin. Images, keypoint order, and visibility were preserved. Some wire points are free ends, not breadboard insertion points. The ZIP includes provenance, split manifest, and correction log.

Source: [BME Electronics, Circuit Detection v5](https://universe.roboflow.com/bme-electronics/circuit-detection-eoirn/dataset/5), [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Adaptations: split, packaging, box expansion. No endorsement implied.

In [ ]:
%pip -q install ultralytics==8.4.156 pyyaml
import torch, ultralytics
print('Ultralytics:', ultralytics.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU connected')
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU, then rerun this cell.'

## Upload the prepared ZIP
Run this cell, choose **breadboard-pilot-ready.zip**, and wait for the upload to finish. Extraction uses a fresh folder each time, so rerunning will not mix data.

In [ ]:
from google.colab import files
from pathlib import Path
import io, zipfile, tempfile, stat, json, hashlib, math
import yaml
from PIL import Image
uploaded = files.upload()
assert len(uploaded) == 1, 'Choose exactly one ZIP file.'
filename, content = next(iter(uploaded.items()))
assert filename.lower().endswith('.zip'), 'Choose the prepared ZIP.'
root = Path(tempfile.mkdtemp(prefix='breadboard_pilot_', dir='/content'))
with zipfile.ZipFile(io.BytesIO(content)) as archive:
    assert sum(e.file_size for e in archive.infolist()) < 200_000_000, 'Unexpected archive size.'
    for entry in archive.infolist():
        assert (root / entry.filename).resolve().is_relative_to(root.resolve()), 'Unsafe archive path.'
        assert not stat.S_ISLNK(entry.external_attr >> 16), 'Symlinks not permitted.'
    archive.extractall(root)
assert (root/'audit.json').exists(), 'This is not the prepared pilot ZIP. Upload breadboard-pilot-ready.zip.'
config = yaml.safe_load((root/'data.yaml').read_text())
assert config['names'] == ['breadboard','resistor','wire'] and config['kpt_shape'] == [8,3], 'Unexpected class or keypoint mapping.'
config['path'] = str(root)
yaml_path = root/'data.yaml'
yaml_path.write_text(yaml.safe_dump(config))
audit = json.loads((root/'audit.json').read_text())
print('Images:', audit['images'])
print('Objects:', audit['objects'])
print('Ready:', root)

In [ ]:
from collections import Counter
seen = set()
for split, expected in [('train',21),('val',3),('test',3)]:
    images = sorted((root/'images'/split).glob('*.jpg'))
    assert len(images) == expected, f'Unexpected image count in {split}.'
    support = Counter()
    for image_path in images:
        digest = hashlib.sha256(image_path.read_bytes()).hexdigest()
        assert digest not in seen, 'Duplicate image detected.'
        seen.add(digest)
        with Image.open(image_path) as image:
            image.verify()
        label = root/'labels'/split/(image_path.stem+'.txt')
        assert label.exists(), f'Missing label: {label.name}'
        for line in label.read_text().splitlines():
            v = list(map(float,line.split()))
            assert len(v) == 29 and all(math.isfinite(x) for x in v), 'Expected 8-keypoint pose labels.'
            assert v[0] in (0,1,2) and 0 < v[3] <= 1 and 0 < v[4] <= 1
            assert 0 <= v[1] <= 1 and 0 <= v[2] <= 1
            for x,y,vis in [v[i:i+3] for i in range(5,29,3)]:
                assert 0 <= x <= 1 and 0 <= y <= 1 and vis in (0,1,2)
            support[config['names'][int(v[0])]] += 1
    print(split, dict(support))
print('Validation passed. There are no held-out wire objects and no LEDs anywhere in this dataset.')

## Train a small pilot model
The first model run downloads pretrained YOLO11 pose weights. The model is adapted to this dataset's 8 keypoints. This 30-epoch run is for checking the workflow, not a promised accuracy target. Keep the browser connected. If GPU memory runs out, change `batch=4` to `batch=2` and rerun.

Flips and mosaic are disabled because these images have ordered board landmarks and very little diversity. Validation selects a checkpoint using the limited validation set; it cannot select for wire quality.

In [ ]:
from ultralytics import YOLO
model = YOLO('yolo11n-pose.pt')
model.train(data=str(yaml_path), epochs=30, imgsz=640, batch=4, device=0, workers=2, seed=42, patience=10, project='/content/circuit_pilot_runs', name='bme_v5', fliplr=0.0, flipud=0.0, mosaic=0.0, degrees=0.0, translate=0.05, scale=0.1)
best = Path(model.trainer.best)
assert best.exists(), 'Training did not produce best.pt.'
print('Training finished:', best)

## Download your model now
Run this cell immediately after training. Keep **breadboard_pilot_best.pt** on your computer so a Colab disconnect cannot lose it. This file is a pilot model, not the final LED detector.

In [ ]:
import shutil
download_path = Path('/content/breadboard_pilot_best.pt')
shutil.copy2(best, download_path)
files.download(str(download_path))

## Inspect held-out predictions
The three test photos show one held-out resistor layout on similar hardware. Their metrics are exploratory, not evidence of performance on arbitrary circuits. They do not measure wire or LED recognition, hole assignment, or electrical fault diagnosis. Keep this test set out of tuning. The generic pose mAP uses default keypoint-distance assumptions; manually inspect terminal errors too.

In [ ]:
trained = YOLO(str(best))
metrics = trained.val(data=str(yaml_path), split='test', imgsz=640, batch=4, device=0, workers=2)
print('Exploratory metrics for the supported test classes:', metrics.results_dict)
from IPython.display import display
for sample in sorted((root/'images'/'test').glob('*.jpg')):
    result = trained.predict(str(sample), conf=0.25, verbose=False)[0]
    print(sample.name)
    display(Image.fromarray(result.plot(kpt_line=False)[..., ::-1]))
summary = {'ultralytics_version':ultralytics.__version__, 'classes':config['names'], 'keypoint_shape':config['kpt_shape'], 'metrics':metrics.results_dict, 'dataset_audit':audit, 'scope':'Pilot breadboard/resistor test only; no wire, LED, connection, or fault accuracy established.'}
summary_path = Path('/content/breadboard_pilot_results.json')
summary_path.write_text(json.dumps(summary, indent=2))
files.download(str(summary_path))

## What to send back
Attach **breadboard_pilot_best.pt** and **breadboard_pilot_results.json** to the project chat, plus a screenshot of the prediction images if helpful.

Next steps: collect independent wire layouts and LED examples, verify insertion-point labels and LED polarity, then build board calibration/hole assignment. The existing two-keypoint LED notebook remains a separate later-stage workflow. Do not upload this pilot ZIP to that notebook.